Importantions of libraries

In [ ]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import yfinance as yf
from typing import Optional
import requests
from datetime import datetime, timedelta
from abc import ABC, abstractmethod
import logging
from typing import List, Dict, Optional, Union
from dataclasses import dataclass, field


Class Stocks

In [ ]:
@dataclass
class Asset(ABC):
    symbol: str
    _prices: Optional[pd.DataFrame] = None

    @abstractmethod
    def get_prices(self) -> pd.DataFrame:
        pass

    @abstractmethod
    def test_symbol(self) -> str:
        pass

@dataclass
class Portfolio:
    name: str
    positions: List[Position] = field(default_factory=list)

    def add_position(self, position: Position):
        if position.quantity <= 0:
            raise ValueError("quantity must be positive")
        self.positions.append(position)

    def del_position(self, position: Position):
        if position not in self.positions:
            raise ValueError("Position not found")
        self.positions.remove(position)

    def total_value(self) -> float:
        return sum(position.total_value() for position in self.positions)

    def total_pnl(self) -> float:
        return sum(position.calculate_pnl() for position in self.positions)

    def __getitem__(self, index: int) -> Position:
        return self.positions[index]

@dataclass
class Position(ABC):
    asset: Asset
    quantity: float = 1

    @abstractmethod
    def calculate_pnl(self) -> float:
        pass

    def total_value(self) -> float:
        return self.asset.get_current_price * self.quantity

@dataclass
class LongPosition(Position):

    entry_price: float = None
    def __post_init__(self):
        if self.entry_price is None:
            self.entry_price = self.asset.get_current_price

    def calculate_pnl(self) -> float:
        return (self.asset.get_current_price - self.entry_price) * self.quantity

@dataclass
class ShortPosition(Position):

    entry_price: float = None

    def __post_init__(self):
        if self.entry_price is None:
            self.entry_price = self.asset.get_current_price

    def calculate_pnl(self) -> float:
        return (self.entry_price - self.asset.get_current_price) * self.quantity

@dataclass
class Crypto(Asset):

    provider: Optional[Provider] = None
    def __post_init__(self):
        if self.provider is None:
            self.provider = YahooProvider()

    def get_prices(self) -> pd.DataFrame:
        if self._prices is None:
            logging.info(f"Fetching prices for {self.symbol}...")
            self._prices = self.provider.get_prices(self.symbol)
        return self._prices

    def test_symbol(self) -> str:
        return self.provider.test_symbol(self.symbol)

    def get_close_series(self) -> pd.Series:
        prices = self.get_prices()
        if 'Close' not in prices.columns:
            raise ValueError("Missing 'Close' column in price data")
        return prices['Close'][self.symbol]

    def get_current_price(self) -> float:
        return self.get_prices()['Close'].iloc[-1]

@dataclass
class Provider(ABC):

    @abstractmethod
    def get_prices(self, symbol: str) -> pd.DataFrame:
        pass

    @abstractmethod
    def test_symbol(self, symbol: str) -> str:
        pass

@dataclass
class YahooProvider(Provider):

    period: str = "1d"
    interval: str = "1m"

    def get_prices(self, symbol: str) -> pd.DataFrame:
        data = yf.download(symbol, period=self.period, interval=self.interval)
        if data.empty:
            raise ValueError(f"Failed to fetch data for {symbol}")
        return data

    def test_symbol(self, symbol: str) -> str:
        try:
            data = self.get_prices(symbol)
            if data.empty:
                raise ValueError
            return symbol
        except Exception:
            raise ValueError(f"Symbol {symbol} not found")

@dataclass
class Indicator(ABC):

    @abstractmethod
    def calculate(self, asset: Asset):
        pass

    def apply_indicator(self, result, indicator=None):
        if indicator is None:
            return result
        return indicator.calculate(result)

    def calculate_with(self, asset, indicator=None):
        result = self.calculate(asset)
        if indicator is not None:
            result = self.apply_indicator(result, indicator)
        return result

@dataclass
class Signal(ABC):

    @abstractmethod
    def generate_signals(self, asset: Asset):
        pass

@dataclass
class SimpleMovingAverage(Indicator):
    period: int = 20

    def calculate(self, asset: Asset | pd.Series) -> pd.Series:
        if hasattr(asset, 'get_close_series'):
            asset = asset.get_close_series()
        result = asset.rolling(window=self.period).mean()
        result.name = f"SMA_{self.period}"
        return result

@dataclass
class ExponentialMovingAverage(Indicator):
    period: int = 20

    def calculate(self, asset: Asset | pd.Series) -> pd.Series:
        if hasattr(asset, 'get_close_series'):
            asset = asset.get_close_series()
        result = asset.ewm(span=self.period, adjust=False).mean()
        result.name = f"EMA_{self.period}"
        return result

@dataclass
class Rsi(Indicator):
    period: int = 14

    def calculate(self, asset: Asset | pd.Series) -> pd.Series:
        if hasattr(asset, 'get_close_series'):
            asset = asset.get_close_series()
        delta = asset.diff()
        gain = (delta.where(delta > 0, 0)).rolling(self.period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(self.period).mean()
        loss = loss.replace(0, np.nan)
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        rsi.name = f"RSI_{self.period}"
        return rsi

@dataclass
class RsiSignal(Signal):
    rsi: Rsi
    buy_threshold: float = 25
    sell_threshold: float = 80

    def generate_signals(self, asset: Asset) -> pd.Series:

        rsi = self.rsi.calculate(asset)
        signals = pd.Series(0, index=rsi.index)
        signals[(rsi < self.buy_threshold)] = -1 #Buy
        signals[(rsi > self.sell_threshold)] = 1 #Sell

        return signals

@dataclass
class StochRsi(Indicator):
    period: int = 14
    smooth: int = 3
    rsi: Rsi = field(default_factory=Rsi)

    def calculate(self, asset: Asset | pd.Series) -> pd.DataFrame:

        rsi_series = self.rsi.calculate(asset)
        rsi_low = rsi_series.rolling(self.period).min()
        rsi_high = rsi_series.rolling(self.period).max()
        denom = (rsi_high - rsi_low).replace(0, np.nan)
        k_raw = (rsi_series - rsi_low) / denom * 100
        k = k_raw.rolling(window=self.smooth).mean() if self.smooth and self.smooth > 1 else k_raw
        d = k.rolling(window=self.smooth).mean() if self.smooth and self.smooth > 1 else k
        k.name = f"Stoch_RSI_K_{self.period}_{self.smooth}"
        d.name = f"Stoch_RSI_D_{self.period}_{self.smooth}"
        return pd.DataFrame({
            "rsi": rsi_series,
            "k": k,
            "d": d
            }).dropna()

@dataclass
class StochRsiSignal(Signal):
    stoch: StochRsi
    buy_threshold: float = 5
    sell_threshold: float = 95

    def generate_signals(self, asset: Asset) -> pd.Series:

        stoch = self.stoch.calculate(asset)
        signals = pd.Series(0, index=stoch.index)
        signals[(stoch['d'] < self.buy_threshold)] = -1 #Buy
        signals[(stoch['d'] > self.sell_threshold)] = 1 #Sell

        return signals

@dataclass
class BollingerBands(Indicator):
    window: int = 20
    std_dev: int = 2

    def calculate(self, asset: Asset | pd.Series) -> pd.Series:

        if hasattr(asset, 'get_close_series'):
            asset = asset.get_close_series()
        price = asset
        middle = price.rolling(self.window).mean()
        std = price.rolling(self.window).std()
        upper = middle + self.std_dev * std
        lower = middle - self.std_dev * std
        bb_percent_b = (price - lower) / (upper - lower)
        bb_percent_b.name = f"BB_percent_b_{self.window}_{self.std_dev}"

        return bb_percent_b

@dataclass
class BollingerBandsSignal(Signal):
    bb: BollingerBands
    buy_threshold: float = 0
    sell_threshold: float = 1

    def generate_signals(self, asset: Asset) -> pd.Series:

        bb = self.bb.calculate(asset)
        signals = pd.Series(0, index=bb.index)
        signals[(bb < self.buy_threshold)] = -1 #Buy
        signals[(bb > self.sell_threshold)] = 1 #Sell

        return signals

@dataclass
class Macd(Indicator):
    fast: int = 12
    slow: int = 26
    signal: int = 9

    def calculate(self, asset: Asset | pd.Series) -> pd.DataFrame:
        if hasattr(asset, 'get_close_series'):
            asset = asset.get_close_series()
        close = asset
        exp1 = close.ewm(span=self.fast).mean()
        exp2 = close.ewm(span=self.slow).mean()
        macd = exp1 - exp2
        signal = macd.ewm(span=self.signal).mean()
        histogram = macd - signal
        macd.name = f"MACD_{self.fast}_{self.slow}_{self.signal}"
        signal.name = f"MACD_Signal_{self.fast}_{self.slow}_{self.signal}"
        histogram.name = f"MACD_Histogram_{self.fast}_{self.slow}_{self.signal}"
        return pd.DataFrame({
        "macd": macd,
        "signal": signal,
        "histogram": histogram
        })


@dataclass
class MacdSignal(Signal):
    macd: Macd

    def generate_signals(self, asset: Asset) -> pd.Series:
        macd = self.macd.calculate(asset)

        macd_line = macd['macd']
        signal_line = macd['signal']

        hist = macd_line - signal_line

        regime = pd.Series(0, index=hist.index)

        regime[(hist > 0) & (hist.shift(1) <= 0)] = 1
        regime[(hist < 0) & (hist.shift(1) >= 0)] = -1

        return regime

@dataclass
class SharpeRatio(Indicator):
    period: int = 52 # Default period for 1wk
    window: int = 60 # Default window for 1wk
    rf: float = 0.01

    def calculate(self, asset: Asset | pd.Series) -> pd.Series:
        if hasattr(asset, 'get_close_series'):
            asset = asset.get_close_series()
        close_prices = asset
        returns = Calculations.log_return(close_prices)
        sharpe_ratio = Calculations.annualized_sharpe(returns, period=self.period, window=self.window, rf=self.rf)
        sharpe_ratio.name = f"Sharpe_{self.period}_{self.window}_{self.rf}"
        return sharpe_ratio

@dataclass
class SharpeSignal(Signal):
    sharpe: SharpeRatio
    buy_threshold: float = -1.5 #Default buy threshold (Sharpe 1wk)
    sell_threshold: float = 2.0 #Default sell threshold (Sharpe 1wk)

    def generate_signals(self, asset: Asset | pd.Series) -> pd.Series:

        sharpe = self.sharpe.calculate(asset)
        signals = pd.Series(0, index=sharpe.index)
        signals[sharpe < self.buy_threshold] = -1
        signals[sharpe > self.sell_threshold] = 1

        return signals
@dataclass
class SortinoRatio(Indicator):
    period: int = 52 # Default period for 1wk
    window: int = 60 # Default window for 1wk
    rf: float = 0.01

    def calculate(self, asset: Asset | pd.Series) -> pd.Series:
        if hasattr(asset, 'get_close_series'):
            asset = asset.get_close_series()
        close_prices = asset
        returns = Calculations.log_return(close_prices)#modificar aqui se quiser sem log! returns = self.create_dataframe_close().pct_change().dropna()
        sortino_ratio = Calculations.annualized_sortino(returns, period=self.period, window=self.window, rf=self.rf)
        sortino_ratio.name = f"Sortino_{self.period}_{self.window}_{self.rf}"
        return sortino_ratio

@dataclass
class SortinoSignal(Signal):
    sortino: SortinoRatio
    buy_threshold: float = -1.5 # Default buy threshold (Sortino 1wk)
    sell_threshold: float = 4.5 # Default sell threshold (Sortino 1wk)

    def generate_signals(self, asset: Asset | pd.Series) -> pd.Series:

        sortino = self.sortino.calculate(asset)
        signals = pd.Series(0, index=sortino.index)
        signals[sortino < self.buy_threshold] = -1
        signals[sortino > self.sell_threshold] = 1

        return signals

class Calculations:

    @staticmethod
    def log_return(prices: pd.Series | Asset) -> pd.Series:
        if hasattr(prices, 'get_close_series'):
            prices = prices.get_close_series()
        returns = np.log(prices / prices.shift(1)).dropna()
        returns.name = 'log_return'
        return returns

    @staticmethod
    def annualized_sharpe(returns: pd.Series, period=52, rf=0.01, window=60) -> pd.Series:
        """
        Calculate annualized Sharpe ratio with rolling window.

        Args:
            returns: Returns series (daily or other frequency)
            period: Number of periods per year (default 252 for trading days)
            rf: Risk-free rate (default 0.01 = 1%)
            window: Rolling window size for calculation

        Returns:
+            pd.Series: Annualized Sharpe ratio with rolling window
+        """
        rolling_mean = returns.rolling(window=window).mean() * period
        rolling_std = returns.rolling(window=window).std() * np.sqrt(period)
        sharpe_ratio = (rolling_mean - rf) / rolling_std
        sharpe_ratio = sharpe_ratio.replace([np.inf, -np.inf], np.nan)
        return sharpe_ratio

    @staticmethod
    def annualized_sortino(returns: pd.Series, period=52, rf=0.01, window=60) -> pd.Series:
        """
+        Calculate annualized Sortino ratio with rolling window.
+
+        Args:
+            returns: Returns series (daily or other frequency)
+            period: Number of periods per year (default 252 for trading days)
+            rf: Risk-free rate (default 0.01 = 1%)
+            window: Rolling window size for calculation
+
+        Returns:
+            pd.Series: Annualized Sortino ratio with rolling window
+        """
        mar = rf / period
        excess = returns - mar
        downside = np.minimum(0, excess)
        downside_deviation = downside.rolling(window=window).std() * np.sqrt(period)
        rolling_mean = excess.rolling(window=window).mean() * period
        sortino_ratio = rolling_mean / downside_deviation
        sortino_ratio = sortino_ratio.replace([np.inf, -np.inf], np.nan)
        return sortino_ratio

@dataclass
class CombinedSignal:
    signals: list = None
    weights: list = None
    quantity_threshold: float = None
    window: int = 1
    min_periods: int = 1

    def generate_signals(self, asset: Asset):

        signals_list = [
            signal.generate_signals(asset)
            for signal in self.signals
        ]

        if self.quantity_threshold is None:
            self.quantity_threshold = len(self.signals)

        df = pd.concat(signals_list, axis=1).fillna(0)

        weights = np.array(self.weights)

        combined = (df * weights).sum(axis=1)

        final_signal = pd.Series(0, index=combined.index)
        final_signal[combined >= self.quantity_threshold] = 1
        final_signal[combined <= -self.quantity_threshold] = -1

        return final_signal

    def confirm_signals(self, asset: Asset):

        signal = self.generate_signals(asset)

        smooth = signal.rolling(
            window=self.window,
            min_periods=self.window
        ).mean()

        final_signal = pd.Series(0, index=signal.index)

        final_signal[smooth >= 1.0] = 1
        final_signal[smooth <= -1.0] = -1

        return final_signal

@dataclass
class Btc(Crypto):
    pass

In [ ]:
#Get BTC data
btc_1_provider = YahooProvider(period="8y", interval="1wk")
btc = Crypto("BTC-USD", provider=btc_1_provider)
btc_monthly_provider = YahooProvider(period="8y", interval="1mo")
btc_monthly = Crypto("BTC-USD", provider=btc_monthly_provider)

In [ ]:
spy_weekly_provider = YahooProvider(period="8y", interval="1wk")
spy_weekly = Crypto("SPY", provider=spy_weekly_provider)
spy_monthly_provider = YahooProvider(period="8y", interval="1mo")
spy_monthly = Crypto("SPY", provider=spy_monthly_provider)

In [ ]:
#Get VIX data Weekly
vix_provider = YahooProvider(period='7y', interval='1wk')
vix = Crypto('^VIX', provider=vix_provider).get_close_series()

#Plot VIX with SMA_9 and SMA_20
vix_sma_9 = ExponentialMovingAverage(period=7).calculate(vix)
vix_sma_20 = ExponentialMovingAverage(period=14).calculate(vix)

#Get VIX data Monthly
vix_monthly_provider = YahooProvider(period='7y', interval='1mo')
vix_monthly = Crypto('^VIX', provider=vix_monthly_provider).get_close_series()
vix_sma_7_monthly = ExponentialMovingAverage(period=7).calculate(vix_monthly)
vix_sma_14_monthly = ExponentialMovingAverage(period=14).calculate(vix_monthly)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [ ]:
sma_200 = SimpleMovingAverage(period=200)
sma_100 = SimpleMovingAverage(period=100)
sma_50 = SimpleMovingAverage(period=50)
sma_20 = SimpleMovingAverage(period=20)

ema_200 = ExponentialMovingAverage(period=200)
ema_100 = ExponentialMovingAverage(period=100)
ema_50 = ExponentialMovingAverage(period=50)

btc_sma_200 = sma_200.calculate(btc)
btc_sma_100 = sma_100.calculate(btc)
btc_sma_50 = sma_50.calculate(btc)

[*********************100%***********************]  1 of 1 completed


In [ ]:
btc_ema_200 = ema_200.calculate(btc)
btc_ema_100 = ema_100.calculate(btc)
btc_ema_50 = ema_50.calculate(btc)


BTC Weekly RSI

In [ ]:
btc_rsi_weekly = Rsi(period=14) #Used for plot
btc_rsi_values_weekly = btc_rsi_weekly.calculate(btc)
btc_rsi_signals_weekly = RsiSignal(rsi=btc_rsi_weekly, buy_threshold=20, sell_threshold=80)
btc_rsi_signals_weekly_gen = btc_rsi_signals_weekly.generate_signals(btc) #Used for signals
rsi_sma_20 = SimpleMovingAverage(period=20).calculate(btc_rsi_values_weekly) #Used for plot SMA_20
btc_rsi_buy = btc_rsi_signals_weekly_gen == -1
btc_rsi_sell = btc_rsi_signals_weekly_gen == 1

SPY Weekly RSI

In [ ]:
spy_weekly_rsi = Rsi(period=14) #Used for plot
spy_weekly_rsi_values = spy_weekly_rsi.calculate(spy_weekly)
spy_weekly_rsi_signals = RsiSignal(rsi=spy_weekly_rsi, buy_threshold=28, sell_threshold=85)
spy_weekly_rsi_signals_gen = spy_weekly_rsi_signals.generate_signals(spy_weekly) #Used for signals
spy_weekly_rsi_sma_20 = SimpleMovingAverage(period=20).calculate(spy_weekly_rsi_values) #Used for plot SMA_20
spy_weekly_rsi_buy = spy_weekly_rsi_signals_gen == -1
spy_weekly_rsi_sell = spy_weekly_rsi_signals_gen == 1

[*********************100%***********************]  1 of 1 completed


BTC Monthly RSI

In [ ]:
btc_rsi_monthly = Rsi(period=14) #Used for plot
btc_rsi_values_monthly = btc_rsi_monthly.calculate(btc_monthly)
btc_rsi_signals_monthly = RsiSignal(rsi=btc_rsi_monthly, buy_threshold=25, sell_threshold=80)
btc_rsi_signals_monthly_gen = btc_rsi_signals_monthly.generate_signals(btc_monthly) #Used for signals
rsi_sma_20_monthly = SimpleMovingAverage(period=16).calculate(btc_rsi_values_monthly) #Used for plot SMA_20
btc_rsi_buy_monthly = btc_rsi_signals_monthly_gen == -1
btc_rsi_sell_monthly = btc_rsi_signals_monthly_gen == 1

SPY Monthly RSI

In [ ]:
spy_rsi_monthly = Rsi(period=14) #Used for plot
spy_rsi_values_monthly = spy_rsi_monthly.calculate(spy_monthly)
spy_rsi_signals_monthly = RsiSignal(rsi=spy_rsi_monthly, buy_threshold=25, sell_threshold=80)
spy_rsi_signals_monthly_gen = spy_rsi_signals_monthly.generate_signals(spy_monthly) #Used for signals
spy_rsi_sma_20_monthly = SimpleMovingAverage(period=9).calculate(spy_rsi_values_monthly) #Used for plot SMA_20
spy_rsi_buy_monthly = spy_rsi_signals_monthly_gen == -1
spy_rsi_sell_monthly = spy_rsi_signals_monthly_gen == 1

[*********************100%***********************]  1 of 1 completed


BTC Weekly Stochastic RSI

In [ ]:
btc_stochrsi_weekly = StochRsi(period=14, smooth=3) #Used for plot
btc_stochrsi_values_weekly = btc_stochrsi_weekly.calculate(btc)
btc_stochrsi_signals_weekly = StochRsiSignal(stoch=btc_stochrsi_weekly)
btc_stochrsi_signals_weekly_gen = btc_stochrsi_signals_weekly.generate_signals(btc)# Used for signals
btc_stochrsi_buy = btc_stochrsi_signals_weekly_gen == -1
btc_stochrsi_sell = btc_stochrsi_signals_weekly_gen == 1

SPY Weekly Stochastic RSI

In [ ]:
rsi_stochrsi_weekly = StochRsi(period=14, smooth=3) #Used for plot
rsi_stochrsi_values_weekly = rsi_stochrsi_weekly.calculate(spy_weekly)
rsi_stochrsi_signals_weekly = StochRsiSignal(stoch=rsi_stochrsi_weekly, buy_threshold=10, sell_threshold=90)
rsi_stochrsi_signals_weekly_gen = rsi_stochrsi_signals_weekly.generate_signals(spy_weekly)# Used for signals
spy_rsi_stochrsi_buy = rsi_stochrsi_signals_weekly_gen == -1
spy_rsi_stochrsi_sell = rsi_stochrsi_signals_weekly_gen == 1

BTC Monthly Stochastic RSI

In [ ]:
btc_stochrsi_monthly = StochRsi(period=14, smooth=3) #Used for plot
btc_stochrsi_values_monthly = btc_stochrsi_monthly.calculate(btc_monthly)
btc_stochrsi_signals_monthly = StochRsiSignal(stoch=btc_stochrsi_monthly)
btc_stochrsi_signals_monthly_gen = btc_stochrsi_signals_monthly.generate_signals(btc_monthly) # Used for signals
btc_stochrsi_buy_monthly = btc_stochrsi_signals_monthly_gen == -1
btc_stochrsi_sell_monthly = btc_stochrsi_signals_monthly_gen == 1

Spy Monthly Stochastic RSI

In [ ]:
spy_stochrsi_monthly = StochRsi(period=14, smooth=3) #Used for plot
spy_stochrsi_values_monthly = spy_stochrsi_monthly.calculate(spy_monthly)
spy_stochrsi_signals_monthly = StochRsiSignal(stoch=spy_stochrsi_monthly)
spy_stochrsi_signals_monthly_gen = spy_stochrsi_signals_monthly.generate_signals(spy_monthly) # Used for signals
spy_stochrsi_buy_monthly = spy_stochrsi_signals_monthly_gen == -1
spy_stochrsi_sell_monthly = spy_stochrsi_signals_monthly_gen == 1

BTC Weekly BB%B

In [ ]:
btc_bb_weekly = BollingerBands(window=30, std_dev=2) #Used for plot
btc_bb_values_weekly = btc_bb_weekly.calculate(btc)
btc_bb_signals_weekly = BollingerBandsSignal(bb=btc_bb_weekly)
btc_bb_signals_weekly_gen = btc_bb_signals_weekly.generate_signals(btc) #Used for signals
bb_sma_20 = SimpleMovingAverage(period=20).calculate(btc_bb_values_weekly) #Used for plot SMA_20
btc_bb_signals_weekly_gen.value_counts() #Used for count signals
btc_bb_buy = btc_bb_signals_weekly_gen == -1
btc_bb_sell = btc_bb_signals_weekly_gen == 1

SPY Weekly BB%B

In [ ]:
spy_bb_weekly = BollingerBands(window=30, std_dev=2) #Used for plot
spy_bb_values_weekly = spy_bb_weekly.calculate(spy_weekly)
spy_bb_signals_weekly = BollingerBandsSignal(bb=spy_bb_weekly)
spy_bb_signals_weekly_gen = spy_bb_signals_weekly.generate_signals(spy_weekly) #Used for signals
spy_bb_sma_20_weekly = SimpleMovingAverage(period=20).calculate(spy_bb_values_weekly) #Used for plot SMA_20
spy_bb_signals_weekly_gen.value_counts() #Used for count signals
spy_bb_buy_weekly = spy_bb_signals_weekly_gen == -1
spy_bb_sell_weekly = spy_bb_signals_weekly_gen == 1

BTC Monthly BB%B

In [ ]:
btc_bb_monthly = BollingerBands(window=20, std_dev=2) #Used for plot
btc_bb_values_monthly = btc_bb_monthly.calculate(btc_monthly)
btc_bb_signals_monthly = BollingerBandsSignal(bb=btc_bb_monthly, buy_threshold=0.1)
btc_bb_signals_monthly_gen = btc_bb_signals_monthly.generate_signals(btc_monthly) #Used for signals
bb_sma_20_monthly = SimpleMovingAverage(period=20).calculate(btc_bb_values_monthly) #Used for plot SMA_20
btc_bb_buy_monthly = btc_bb_signals_monthly_gen == -1
btc_bb_sell_monthly = btc_bb_signals_monthly_gen == 1

Spy Monthly BB%B

In [ ]:
spy_bb_monthly = BollingerBands(window=20, std_dev=2) #Used for plot
spy_bb_values_monthly = spy_bb_monthly.calculate(spy_monthly)
spy_bb_signals_monthly = BollingerBandsSignal(bb=spy_bb_monthly, buy_threshold=0.1)
spy_bb_signals_monthly_gen = spy_bb_signals_monthly.generate_signals(spy_monthly) #Used for signals
spy_bb_sma_20_monthly = SimpleMovingAverage(period=9).calculate(spy_bb_values_monthly) #Used for plot SMA_20
spy_bb_buy_monthly = spy_bb_signals_monthly_gen == -1
spy_bb_sell_monthly = spy_bb_signals_monthly_gen == 1

BTC Weekly MACD

In [ ]:
btc_macd = Macd()#Used for plot
btc_macd_values_weekly = btc_macd.calculate(btc)
btc_macd_signals_weekly = MacdSignal(btc_macd).generate_signals(btc)#Used for signals
btc_macd_buy = btc_macd_signals_weekly == 1
btc_macd_sell = btc_macd_signals_weekly == -1

Spy Weekly MACD

In [ ]:
spy_macd = Macd()#Used for plot
spy_macd_values_weekly = spy_macd.calculate(spy_weekly)
spy_macd_signals_weekly = MacdSignal(spy_macd).generate_signals(spy_weekly)#Used for signals
spy_macd_buy_weekly = spy_macd_signals_weekly == 1
spy_macd_sell_weekly = spy_macd_signals_weekly == -1

BTC Monthly MACD

In [ ]:
btc_macd_monthly = Macd()#Used for plot
btc_macd_values_monthly = btc_macd_monthly.calculate(btc_monthly)
btc_macd_signals_monthly = MacdSignal(btc_macd_monthly).generate_signals(btc_monthly)#Used for signals
btc_macd_buy_monthly = btc_macd_signals_monthly == 1
btc_macd_sell_monthly = btc_macd_signals_monthly == -1

Spy Monthly MACD

In [ ]:
spy_macd_monthly = Macd()#Used for plot
spy_macd_values_monthly = spy_macd_monthly.calculate(spy_monthly)
spy_macd_signals_monthly = MacdSignal(spy_macd_monthly).generate_signals(spy_monthly)#Used for signals
spy_macd_buy_monthly = spy_macd_signals_monthly == 1
spy_macd_sell_monthly = spy_macd_signals_monthly == -1

BTC Monthly Sharpe

In [ ]:
#Used class SharpeSignal Default Settings
btc_sharpe_monthly = SharpeRatio(period=12, window=6)
btc_sharpe_values_monthly = btc_sharpe_monthly.calculate(btc_monthly)
btc_sharpe_signals_monthly = SharpeSignal(sharpe=btc_sharpe_monthly, buy_threshold=-2.0, sell_threshold=3.5)
btc_sharpe_signals_monthly_gen = btc_sharpe_signals_monthly.generate_signals(btc_monthly)

btc_sharpe_sma_14_monthly = SimpleMovingAverage(period=14).calculate(btc_sharpe_values_monthly)#Used for plot SMA_14 in sharpe

sharpe_buy_monthly = btc_sharpe_signals_monthly_gen == -1
sharpe_sell_monthly = btc_sharpe_signals_monthly_gen == 1

Spy Monthly Sharpe

In [ ]:
#Used class SharpeSignal Default Settings
spy_sharpe_monthly = SharpeRatio(period=12, window=6)
spy_sharpe_values_monthly = spy_sharpe_monthly.calculate(spy_monthly)
spy_sharpe_signals_monthly = SharpeSignal(sharpe=spy_sharpe_monthly, buy_threshold=-0.5, sell_threshold=3.5)
spy_sharpe_signals_monthly_gen = spy_sharpe_signals_monthly.generate_signals(spy_monthly)

spy_sharpe_sma_14_monthly = SimpleMovingAverage(period=14).calculate(spy_sharpe_values_monthly)#Used for plot SMA_14 in sharpe

spy_sharpe_buy_monthly = spy_sharpe_signals_monthly_gen == -1
spy_sharpe_sell_monthly = spy_sharpe_signals_monthly_gen == 1

BTC Weekly Sharpe

In [ ]:
#Used class SharpeSignal Default Settings
btc_sharpe = SharpeRatio(period=52, window=60)
btc_sharpe_values = btc_sharpe.calculate(btc)
btc_sharpe_signals = SharpeSignal(sharpe=btc_sharpe)
btc_sharpe_signals_gen = btc_sharpe_signals.generate_signals(btc)

btc_sharpe_sma_20 = SimpleMovingAverage(period=20).calculate(btc_sharpe_values)#Used for plot SMA_20 in sharpe

btc_sharpe_buy_weekly = btc_sharpe_signals_gen == -1
btc_sharpe_sell_weekly = btc_sharpe_signals_gen == 1

Spy Weekly Sharpe

In [ ]:
#Used class SharpeSignal Default Settings
spy_sharpe = SharpeRatio(period=52, window=60)
spy_sharpe_values = spy_sharpe.calculate(spy_weekly)
spy_sharpe_signals = SharpeSignal(sharpe=spy_sharpe, buy_threshold=-0.5, sell_threshold=2.19)
spy_sharpe_signals_gen = spy_sharpe_signals.generate_signals(spy_weekly)

spy_sharpe_sma_20_weekly = SimpleMovingAverage(period=20).calculate(spy_sharpe_values)#Used for plot SMA_20 in sharpe

spy_sharpe_buy_weekly = spy_sharpe_signals_gen == -1
spy_sharpe_sell_weekly = spy_sharpe_signals_gen == 1

BTC Weekly Sortino

In [ ]:
#Used class SortinoSignal Default Settings
btc_sortino = SortinoRatio(period=52, window=60)
btc_sortino_values = btc_sortino.calculate(btc)
btc_sortino_signals = SortinoSignal(sortino=btc_sortino, sell_threshold=4.9).generate_signals(btc)

btc_sortino_sma_14 = SimpleMovingAverage(period=14).calculate(btc_sortino_values)#Used for plot SMA_20 in sortino
btc_sortino_sma_7 = SimpleMovingAverage(period=7).calculate(btc_sortino_values)#Used for plot SMA_9 in sortino

sortino_buy_weekly = btc_sortino_signals == -1
sortino_sell_weekly = btc_sortino_signals == 1

Spy Weekly Sortino

In [ ]:
#Used class SortinoSignal Default Settings
spy_sortino = SortinoRatio(period=52, window=60)
spy_sortino_values = spy_sortino.calculate(spy_weekly)
spy_sortino_signals = SortinoSignal(sortino=spy_sortino, buy_threshold=-0.7, sell_threshold=4.7)
spy_sortino_signals_gen = spy_sortino_signals.generate_signals(spy_weekly)

spy_sortino_sma_14_weekly = SimpleMovingAverage(period=14).calculate(spy_sortino_values)#Used for plot SMA_20 in sortino
spy_sortino_sma_7_weekly = SimpleMovingAverage(period=7).calculate(spy_sortino_values)#Used for plot SMA_9 in sortino

spy_sortino_buy_weekly = spy_sortino_signals_gen == -1
spy_sortino_sell_weekly = spy_sortino_signals_gen == 1

BTC Monthly Sortino

In [ ]:
#Used class SortinoSignal Default Settings
btc_sortino_monthly = SortinoRatio(period=12, window=14)
btc_sortino_values_monthly = btc_sortino_monthly.calculate(btc_monthly)
btc_sortino_signals_monthly = SortinoSignal(sortino=btc_sortino_monthly, buy_threshold=-1.5, sell_threshold=6.0).generate_signals(btc_monthly)

btc_sortino_sma_14_monthly = SimpleMovingAverage(period=5).calculate(btc_sortino_values_monthly)#Used for plot SMA_20 in sortino
btc_sortino_sma_7_monthly = SimpleMovingAverage(period=3).calculate(btc_sortino_values_monthly)#Used for plot SMA_9 in sortino

sortino_buy_monthly = btc_sortino_signals_monthly == -1
sortino_sell_monthly = btc_sortino_signals_monthly == 1

Spy Monthly Sortino

In [ ]:
#Used class SortinoSignal Default Settings
spy_sortino_monthly = SortinoRatio(period=12, window=20)
spy_sortino_values_monthly = spy_sortino_monthly.calculate(spy_monthly)
spy_sortino_signals_monthly = SortinoSignal(sortino=spy_sortino_monthly, buy_threshold=-1.0, sell_threshold=6.5).generate_signals(spy_monthly)

spy_sortino_sma_14_monthly = SimpleMovingAverage(period=5).calculate(spy_sortino_values_monthly)#Used for plot SMA_20 in sortino
spy_sortino_sma_7_monthly = SimpleMovingAverage(period=3).calculate(spy_sortino_values_monthly)#Used for plot SMA_9 in sortino

spy_sortino_buy_monthly = spy_sortino_signals_monthly == -1
spy_sortino_sell_monthly = spy_sortino_signals_monthly == 1

BTC Weekly Combined

In [ ]:
#Used class CombinedSignal
btc_df = btc.get_prices()

btc_combined = CombinedSignal(signals = [btc_bb_signals_weekly, btc_stochrsi_signals_weekly,btc_rsi_signals_weekly], weights = [1.0, 1.0, 1.0], quantity_threshold=3.0, window=2, min_periods=2)
btc_combined_signals = btc_combined.confirm_signals(btc)
btc_combined_signals = btc_combined_signals.reindex(btc_df.index).fillna(0) #Used fo plot in BTC-USD candlestick

buy_combined_weekly = btc_combined_signals == -1
sell_combined_weekly = btc_combined_signals == 1

SPY Weekly Combined

In [ ]:
#Used class CombinedSignal
spy_weekly_df = spy_weekly.get_prices()

spy_combined = CombinedSignal(signals = [spy_bb_signals_weekly, rsi_stochrsi_signals_weekly, spy_weekly_rsi_signals], weights = [1.0, 1.0, 1.0], quantity_threshold=3.0, window=2, min_periods=2)
spy_combined_signals = spy_combined.confirm_signals(spy_weekly)
spy_combined_signals = spy_combined_signals.reindex(spy_weekly_df.index).fillna(0) #Used fo plot in BTC-USD candlestick

spy_buy_combined_weekly = spy_combined_signals == -1
spy_sell_combined_weekly = spy_combined_signals == 1

BTC Monthly Combined Strategy

In [ ]:
#Used class CombinedSignal
btc_df_monthly = btc_monthly.get_prices()

btc_combined_monthly = CombinedSignal(signals = [btc_rsi_signals_monthly, btc_stochrsi_signals_monthly, btc_bb_signals_monthly], weights = [1.0, 0.5, 1.0], quantity_threshold=2.0)
btc_combined_signals_monthly = btc_combined_monthly.generate_signals(btc_monthly)
btc_combined_signals_monthly = btc_combined_signals_monthly.reindex(btc_df_monthly.index).fillna(0) #Used fo plot in BTC-USD candlestick

buy_combined_monthly = btc_combined_signals_monthly == -1
sell_combined_monthly = btc_combined_signals_monthly == 1

Spy Monthly Combined Strategy

In [ ]:
#Used class CombinedSignal
spy_df_monthly = spy_monthly.get_prices()

spy_combined_monthly = CombinedSignal(signals = [spy_rsi_signals_monthly, spy_stochrsi_signals_monthly, spy_bb_signals_monthly], weights = [1.0, 0.5, 1.0], quantity_threshold=2.0)
spy_combined_signals_monthly = spy_combined_monthly.generate_signals(spy_monthly)
spy_combined_signals_monthly = spy_combined_signals_monthly.reindex(spy_df_monthly.index).fillna(0) #Used fo plot in SPY candlestick

spy_buy_combined_monthly = spy_combined_signals_monthly == -1
spy_sell_combined_monthly = spy_combined_signals_monthly == 1

BTC Weekly Comfirmed Strategy

In [ ]:
btc_comfirmed = CombinedSignal(signals = [SharpeSignal(sharpe=btc_sharpe, buy_threshold=-1.5, sell_threshold=2.1), SortinoSignal(sortino=btc_sortino, buy_threshold=-1.7, sell_threshold=4.7)], weights = [1.0, 1.0], quantity_threshold=2.0, window=1, min_periods=1)
btc_confirmed = btc_comfirmed.confirm_signals(btc)
btc_confirmed = btc_confirmed.reindex(btc_df.index).fillna(0)
buy_confirmed_weekly = btc_confirmed == -1
sell_confirmed_weekly = btc_confirmed == 1

Spy Weekly Confirmed Strategy

In [ ]:
spy_comfirmed = CombinedSignal(signals = [SharpeSignal(sharpe=spy_sharpe, buy_threshold=-0.5, sell_threshold=2.0), SortinoSignal(sortino=spy_sortino, buy_threshold=-1.2, sell_threshold=4.7)], weights = [1.0, 1.0], quantity_threshold=2.0, window=1, min_periods=1)
spy_confirmed = spy_comfirmed.confirm_signals(spy_weekly)
spy_confirmed = spy_confirmed.reindex(spy_weekly_df.index).fillna(0)
spy_buy_confirmed_weekly = spy_confirmed == -1
spy_sell_confirmed_weekly = spy_confirmed == 1

BTC Monthly Confirmed Strategy

In [ ]:
btc_comfirmed_monthly = CombinedSignal(signals = [SharpeSignal(sharpe=btc_sharpe_monthly, buy_threshold=-1.5, sell_threshold=3.5), SortinoSignal(sortino=btc_sortino_monthly, buy_threshold=-1.5, sell_threshold=5.0)], weights = [1.0, 1.0], quantity_threshold=2.0, window=2, min_periods=1)
btc_confirmed_monthly = btc_comfirmed_monthly.confirm_signals(btc_monthly)
btc_confirmed_monthly = btc_confirmed_monthly.reindex(btc_df_monthly.index).fillna(0)
buy_sharpe_sortino_confirmed_monthly = btc_confirmed_monthly == -1
sell_sharpe_sortino_confirmed_monthly = btc_confirmed_monthly == 1

Spy Monthly Confirmed Strategy

In [ ]:
spy_comfirmed_monthly = CombinedSignal(signals = [SharpeSignal(sharpe=spy_sharpe_monthly), SortinoSignal(sortino=spy_sortino_monthly)], weights = [1.0, 1.0], quantity_threshold=2.0, window=2, min_periods=1)
spy_confirmed_monthly = spy_comfirmed_monthly.confirm_signals(spy_monthly)
spy_confirmed_monthly = spy_confirmed_monthly.reindex(spy_df_monthly.index).fillna(0)
spy_buy_sharpe_sortino_confirmed_monthly = spy_confirmed_monthly == -1
spy_sell_sharpe_sortino_confirmed_monthly = spy_confirmed_monthly == 1

Figure BTC Weekly

In [ ]:
fig_btc_weekly = make_subplots(
    rows=6, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.0008,
    subplot_titles=('BTC-USD'),
    row_width=[1.0, 1.0, 1.0, 1.0, 1.0, 2.0]
)

fig_btc_weekly.add_trace(go.Candlestick(
    x=btc_df.index,
    open=btc_df['Open']['BTC-USD'],
    high=btc_df['High']['BTC-USD'],
    low=btc_df['Low']['BTC-USD'],
    close=btc_df['Close']['BTC-USD'],
    name='BTC-USD',

), row=1, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_df.index[buy_confirmed_weekly],
    y=btc_df['Close']['BTC-USD'][buy_confirmed_weekly],
    mode='markers',
    marker=dict(size=15,
                color='#00BFFF',
                symbol='star'),
    showlegend=True,
    name='Buy Confirmed Weekly'
), row=1, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_df.index[sell_confirmed_weekly],
    y=btc_df['Close']['BTC-USD'][sell_confirmed_weekly],
    mode='markers',
    marker=dict(size=15,
                color='#FFFF00',
                colorscale='PuRd',
                symbol='star'),
    showlegend=True,
    name='Sell Confirmed Weekly'
), row=1, col=1)

'''fig_btc_weekly.add_trace(go.Scatter(
    x=btc_df.index[buy_combined_weekly],
    y=btc_df['Close']['BTC-USD'][buy_combined_weekly],
    mode='markers',
    marker=dict(size=15,
                color='blue',
                colorscale='PuRd',
                symbol='arrow-up'),
    opacity=0.7,
    showlegend=True,
    name='Buy Combined Signals'
), row=1, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_df.index[sell_combined_weekly],
    y=btc_df['Close']['BTC-USD'][sell_combined_weekly],
    mode='markers',
    marker=dict(size=15,
                color='red',
                colorscale='PuRd',
                symbol='arrow-down'),
    opacity=0.7,
    showlegend=True,
    name='Sell Combined Signals'
), row=1, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_df.index[btc_macd_buy],
    y=btc_df['Close']['BTC-USD'][btc_macd_buy],
    mode='markers',
    marker=dict(size=12,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='MACD Buy Signals'
), row=1, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_df.index[btc_macd_sell],
    y=btc_df['Close']['BTC-USD'][btc_macd_sell],
    mode='markers',
    marker=dict(size=12,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='MACD Sell Signals'
), row=1, col=1)'''

#---------------------------RSI------------------------#

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_rsi_values_weekly.index,
    y=btc_rsi_values_weekly,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='RSI'
), row=2, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_rsi_values_weekly[btc_rsi_buy].index,
    y=btc_rsi_values_weekly[btc_rsi_buy],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='RSI Buy Signals'
), row=2, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_rsi_values_weekly[btc_rsi_sell].index,
    y=btc_rsi_values_weekly[btc_rsi_sell],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='RSI Sell Signals'
), row=2, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=rsi_sma_20.index,
    y=rsi_sma_20,
    line=dict(color='orange', width=2),
    opacity=0.7,
    showlegend=True,
    name='SMA 20'
), row=2, col=1)

fig_btc_weekly.add_hline(y=20, line_dash="dot", line_color="lightgreen", row=2, col=1)
fig_btc_weekly.add_hline(y=50, line_dash="dot", line_color="gray", row=2, col=1)
fig_btc_weekly.add_hline(y=80, line_dash="dot", line_color="lightcoral", row=2, col=1)


#--------------------------STOCH RSI---------------------#

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_stochrsi_values_weekly['d'].index,
    y=btc_stochrsi_values_weekly['d'],
    line=dict(color='lightcoral', width=2),
    showlegend=True,
    name='StochRSI d'
), row=3, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_rsi_values_weekly.index,
    y=btc_rsi_values_weekly,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='RSI'
), row=3, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_stochrsi_values_weekly['d'][btc_stochrsi_buy].index,
    y=btc_stochrsi_values_weekly['d'][btc_stochrsi_buy],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='StochRSI Buy Signals'
), row=3, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_stochrsi_values_weekly['d'][btc_stochrsi_sell].index,
    y=btc_stochrsi_values_weekly['d'][btc_stochrsi_sell],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='StochRSI Sell Signals'
), row=3, col=1)

fig_btc_weekly.add_hline(y=5, line_dash="dot", line_color="lightgreen", row=3, col=1)
fig_btc_weekly.add_hline(y=50, line_dash="dot", line_color="gray", row=3, col=1)
fig_btc_weekly.add_hline(y=90, line_dash="dot", line_color="lightcoral", row=3, col=1)


#--------------------BB%B-----------------------#

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_bb_values_weekly.index,
    y=btc_bb_values_weekly,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='BB'
), row=4, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_bb_values_weekly[btc_bb_buy].index,
    y=btc_bb_values_weekly[btc_bb_buy],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='BB Buy Signals'
), row=4, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_bb_values_weekly[btc_bb_sell].index,
    y=btc_bb_values_weekly[btc_bb_sell],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='BB Sell Signals'
), row=4, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=bb_sma_20.index,
    y=bb_sma_20,
    line=dict(color='orange', width=2),
    opacity=0.7,
    showlegend=True,
    name='BB SMA 20'
), row=4, col=1)

fig_btc_weekly.add_hline(y=0, line_dash="dot", line_color="lightgreen", row=4, col=1)
fig_btc_weekly.add_hline(y=0.50, line_dash="dot", line_color="gray", row=4, col=1)
fig_btc_weekly.add_hline(y=1, line_dash="dot", line_color="lightcoral", row=4, col=1)

#------------------------MACD------------------------#

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_macd_values_weekly['macd'].index,
    y=btc_macd_values_weekly['macd'],
    line=dict(color='lightblue', width=1),
    showlegend=True,
    name='MACD'
), row=5, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_macd_values_weekly['signal'].index,
    y=btc_macd_values_weekly['signal'],
    line=dict(color='red', width=1),
    opacity=0.7,
    showlegend=True,
    name='MACD Signal'
), row=5, col=1)

fig_btc_weekly.add_trace(go.Bar(
    x=btc_macd_values_weekly['histogram'].index,
    y=btc_macd_values_weekly['histogram'],
    marker_color=btc_macd_values_weekly['histogram'],
    showlegend=True,
    name='MACD Histogram'
), row=5, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_macd_values_weekly['macd'][btc_macd_buy].index,
    y=btc_macd_values_weekly['macd'][btc_macd_buy],
    mode='markers',
    marker=dict(size=12,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='MACD Buy Signals'
), row=5, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_macd_values_weekly['macd'][btc_macd_sell].index,
    y=btc_macd_values_weekly['macd'][btc_macd_sell],
    mode='markers',
    marker=dict(size=12,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='MACD Sell Signals'
), row=5, col=1)


#------------------------SHARPE-----------------------#


fig_btc_weekly.add_trace(go.Scatter(
    x=btc_sharpe_values.index,
    y=btc_sharpe_values,
    line=dict(color='lightblue', width=1),
    showlegend=True,
    name='Sharpe'
), row=6, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_sharpe_values[btc_sharpe_buy_weekly].index,
    y=btc_sharpe_values[btc_sharpe_buy_weekly],
    mode='markers',
    marker=dict(size=btc_sharpe_values[btc_sharpe_buy_weekly].abs()*7,
                color='green',
                symbol='circle'),

    showlegend=True,
    opacity=0.5,
    name='Sharpe Buy Signals'
), row=6, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_sharpe_values[btc_sharpe_sell_weekly].index,
    y=btc_sharpe_values[btc_sharpe_sell_weekly],
    mode='markers',
    marker=dict(size=btc_sharpe_values[btc_sharpe_sell_weekly].abs()*4,
                color='red',
                symbol='circle'),

    showlegend=True,
    opacity=0.7,
    name='Sharpe Sell Signals'
), row=6, col=1)

'''fig_btc_weekly.add_trace(go.Scatter(
    x=btc_sharpe_sma_20.index,
    y=btc_sharpe_sma_20,
    line=dict(color='purple', width=2),
    showlegend=True,
    opacity=0.7,
    name='Sharpe SMA 20'
), row=6, col=1)'''

#------------------------SORTINO-----------------------#

fig_btc_weekly.add_trace(go.Bar(
    x=btc_sortino_values.index,
    y=btc_sortino_values,
    marker_color=btc_sortino_values,
    showlegend=True,
    opacity=0.7,
    name='Sortino'
), row=6, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_sortino_values[sortino_buy_weekly].index,
    y=btc_sortino_values[sortino_buy_weekly],
    mode='markers',
    marker=dict(size=btc_sortino_values[sortino_buy_weekly].abs()*5,
                color='green',
                symbol='triangle-up'),

    showlegend=True,
    opacity=0.7,
    name='Sortino Buy Signals'
), row=6, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_sortino_values[sortino_sell_weekly].index,
    y=btc_sortino_values[sortino_sell_weekly],
    mode='markers',
    marker=dict(size=btc_sortino_values[sortino_sell_weekly].abs()*3,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='Sortino Sell Signals'
), row=6, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_sortino_sma_14.index,
    y=btc_sortino_sma_14,
    line=dict(color='#FFBF00', width=2),
    showlegend=True,
    opacity=0.5,
    name='Sortino SMA 14'
), row=6, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=btc_sortino_sma_7.index,
    y=btc_sortino_sma_7,
    line=dict(color='#FF4500', width=2),
    showlegend=True,
    opacity=0.5,
    name='Sortino SMA 7'
), row=6, col=1)

#------------------------VIX----------------------#

'''fig_btc_weekly.add_trace(go.Bar(
    x=vix.index,
    y=vix,
    marker_color=vix,
    showlegend=True,
    opacity=0.7,
    name='VIX'
), row=7, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=vix_sma_9.index,
    y=vix_sma_9,
    marker_color='orange',
    showlegend=True,
    opacity=0.7,
    name='VIX SMA 9'
), row=7, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=vix_sma_20.index,
    y=vix_sma_20,
    marker_color='lightgreen',
    showlegend=True,
    name='VIX SMA 50'
), row=7, col=1)'''

fig_btc_weekly.update_layout(xaxis_rangeslider_visible=False, template='plotly_dark', height=1150, width=1600)

Figure BTC Monthly

In [ ]:
fig_btc_monthly = make_subplots(
    rows=6, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.008,
    subplot_titles=('BTC-USD'),
    row_width=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    specs=[[{"secondary_y": True}], [{"secondary_y": True}], [{"secondary_y": True}], [{"secondary_y": True}], [{"secondary_y": True}], [{"secondary_y": True}]]
)

fig_btc_monthly.add_trace(go.Candlestick(
    x=btc_df_monthly.index,
    open=btc_df_monthly['Open']['BTC-USD'],
    high=btc_df_monthly['High']['BTC-USD'],
    low=btc_df_monthly['Low']['BTC-USD'],
    close=btc_df_monthly['Close']['BTC-USD'],
    name='BTC-USD',

), row=1, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_df_monthly.index[buy_sharpe_sortino_confirmed_monthly],
    y=btc_df_monthly['Close']['BTC-USD'][buy_sharpe_sortino_confirmed_monthly],
    mode='markers',
    marker=dict(size=15,
                color='#00BFFF',
                symbol='star'),
    showlegend=True,
    name='Buy Very Strong Signals'
), row=1, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_df_monthly.index[sell_sharpe_sortino_confirmed_monthly],
    y=btc_df_monthly['Close']['BTC-USD'][sell_sharpe_sortino_confirmed_monthly],
    mode='markers',
    marker=dict(size=15,
                color='#FFFF00',
                symbol='star'),
    showlegend=True,
    name='Sell Very Strong Signals'
), row=1, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_df_monthly.index[buy_combined_monthly],
    y=btc_df_monthly['Close']['BTC-USD'][buy_combined_monthly],
    mode='markers',
    marker=dict(size=10,
                color='blue',
                colorscale='PuRd',
                symbol='arrow-up'),
    opacity=0.7,
    showlegend=True,
    name='Buy Strong Signals'
), row=1, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_df_monthly.index[sell_combined_monthly],
    y=btc_df_monthly['Close']['BTC-USD'][sell_combined_monthly],
    mode='markers',
    marker=dict(size=10,
                color='red',
                colorscale='PuRd',
                symbol='arrow-down'),
    opacity=0.7,
    showlegend=True,
    name='Sell Strong Signals'
), row=1, col=1)


#------------------------MACD------------------------#


fig_btc_monthly.add_trace(go.Scatter(
    x=btc_df_monthly.index[btc_macd_buy_monthly],
    y=btc_df_monthly['Close']['BTC-USD'][btc_macd_buy_monthly],
    mode='markers',
    marker=dict(size=15,
                color='#4CAF50',
                symbol='arrow-up'),
    showlegend=True,
    name='MACD Buy Signals Monthly'
), row=1, col=1)


fig_btc_monthly.add_trace(go.Scatter(
    x=btc_df_monthly.index[btc_macd_sell_monthly],
    y=btc_df_monthly['Close']['BTC-USD'][btc_macd_sell_monthly],
    mode='markers',
    marker=dict(size=15,
                color='#FF0000',
                symbol='arrow-down'),
    showlegend=True,
    name='MACD Sell Signals Monthly'
), row=1, col=1)


#---------------------------------Sharpe Sortino -------------------------------------#


fig_btc_monthly.add_trace(go.Scatter(
    x=btc_sharpe_values_monthly.index,
    y=btc_sharpe_values_monthly,
    line=dict(color='lightblue', width=1),
    showlegend=True,
    name='Sharpe'
), row=2, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_sharpe_values_monthly[sharpe_buy_monthly].index,
    y=btc_sharpe_values_monthly[sharpe_buy_monthly],
    mode='markers',
    marker=dict(size=btc_sharpe_values_monthly[sharpe_buy_monthly].abs()*7,
                color='green',
                symbol='circle'),

    showlegend=True,
    opacity=0.5,
    name='Sharpe Buy Signals'
), row=2, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_sharpe_values_monthly[sharpe_sell_monthly].index,
    y=btc_sharpe_values_monthly[sharpe_sell_monthly],
    mode='markers',
    marker=dict(size=btc_sharpe_values_monthly[sharpe_sell_monthly].abs()*4,
                color='red',
                symbol='circle'),

    showlegend=True,
    opacity=0.7,
    name='Sharpe Sell Signals'
), row=2, col=1)

'''fig_btc_monthly.add_trace(go.Scatter(
    x=btc_sharpe_sma_14_monthly.index,
    y=btc_sharpe_sma_14_monthly,
    line=dict(color='purple', width=2),
    showlegend=True,
    opacity=0.7,
    name='Sharpe SMA 14'
), row=2, col=1)'''

fig_btc_monthly.add_trace(go.Bar(
    x=btc_sortino_values_monthly.index,
    y=btc_sortino_values_monthly,
    marker_color=btc_sortino_values_monthly,
    showlegend=True,
    opacity=0.7,
    name='Sortino'
), row=2, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_sortino_values_monthly[sortino_buy_monthly].index,
    y=btc_sortino_values_monthly[sortino_buy_monthly],
    mode='markers',
    marker=dict(size=btc_sortino_values_monthly[sortino_buy_monthly].abs()*5,
                color='green',
                symbol='triangle-up'),

    showlegend=True,
    opacity=0.7,
    name='Sortino Buy Signals'
), row=2, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_sortino_values_monthly[sortino_sell_monthly].index,
    y=btc_sortino_values_monthly[sortino_sell_monthly],
    mode='markers',
    marker=dict(size=btc_sortino_values_monthly[sortino_sell_monthly].abs()*3,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='Sortino Sell Signals'
), row=2, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_sortino_sma_14_monthly.index,
    y=btc_sortino_sma_14_monthly,
    line=dict(color='#FFBF00', width=2),
    showlegend=True,
    opacity=0.5,
    name='Sortino SMA 14'
), row=2, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_sortino_sma_7_monthly.index,
    y=btc_sortino_sma_7_monthly,
    line=dict(color='#FF4500', width=2),
    showlegend=True,
    opacity=0.5,
    name='Sortino SMA 7'
), row=2, col=1)

#----------------------------RSI-----------------------------#


fig_btc_monthly.add_trace(go.Scatter(
    x=btc_rsi_values_monthly.index,
    y=btc_rsi_values_monthly,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='RSI'
), row=3, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_rsi_values_monthly[btc_rsi_buy_monthly].index,
    y=btc_rsi_values_monthly[btc_rsi_buy_monthly],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='RSI Buy Signals'
), row=3, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_rsi_values_monthly[btc_rsi_sell_monthly].index,
    y=btc_rsi_values_monthly[btc_rsi_sell_monthly],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='RSI Sell Signals'
), row=3, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=rsi_sma_20_monthly.index,
    y=rsi_sma_20_monthly,
    line=dict(color='orange', width=2),
    opacity=0.7,
    showlegend=True,
    name='RSI SMA 20'
), row=3, col=1)

fig_btc_monthly.add_hline(y=20, line_dash="dot", line_color="lightgreen", row=3, col=1)
fig_btc_monthly.add_hline(y=50, line_dash="dot", line_color="gray", row=3, col=1)
fig_btc_monthly.add_hline(y=80, line_dash="dot", line_color="lightcoral", row=3, col=1)


#--------------------------------Stoch Rsi------------------------------------#

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_stochrsi_values_monthly['d'].index,
    y=btc_stochrsi_values_monthly['d'],
    line=dict(color='lightcoral', width=2),
    showlegend=True,
    name='StochRSI d'
), row=4, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_rsi_values_monthly.index,
    y=btc_rsi_values_monthly,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='StochRSI'
), row=4, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_stochrsi_values_monthly['d'][btc_stochrsi_buy_monthly].index,
    y=btc_stochrsi_values_monthly['d'][btc_stochrsi_buy_monthly],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='StochRSI Buy Signals'
), row=4, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_stochrsi_values_monthly['d'][btc_stochrsi_sell_monthly].index,
    y=btc_stochrsi_values_monthly['d'][btc_stochrsi_sell_monthly],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='StochRSI Sell Signals'
), row=4, col=1)

fig_btc_monthly.add_hline(y=10, line_dash="dot", line_color="lightgreen", row=4, col=1)
fig_btc_monthly.add_hline(y=50, line_dash="dot", line_color="gray", row=4, col=1)
fig_btc_monthly.add_hline(y=90, line_dash="dot", line_color="lightcoral", row=4, col=1)

#---------------------------------BB%B----------------------------#

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_bb_values_monthly.index,
    y=btc_bb_values_monthly,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='BB%B'
), row=5, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_bb_values_monthly[btc_bb_buy_monthly].index,
    y=btc_bb_values_monthly[btc_bb_buy_monthly],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='BB Buy Signals'
), row=5, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_bb_values_monthly[btc_bb_sell_monthly].index,
    y=btc_bb_values_monthly[btc_bb_sell_monthly],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='BB Sell Signals'
), row=5, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=bb_sma_20_monthly.index,
    y=bb_sma_20_monthly,
    line=dict(color='orange', width=2),
    opacity=0.7,
    showlegend=True,
    name='BB SMA 20'
), row=5, col=1)

fig_btc_monthly.add_hline(y=0, line_dash="dot", line_color="lightgreen", row=5, col=1)
fig_btc_monthly.add_hline(y=0.50, line_dash="dot", line_color="gray", row=5, col=1)
fig_btc_monthly.add_hline(y=1, line_dash="dot", line_color="lightcoral", row=5, col=1)

#------------------------------MACD--------------------------#

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_macd_values_monthly['macd'].index,
    y=btc_macd_values_monthly['macd'],
    line=dict(color='lightblue', width=1),
    showlegend=True,
    name='MACD'
), row=6, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_macd_values_monthly['signal'].index,
    y=btc_macd_values_monthly['signal'],
    line=dict(color='red', width=1),
    opacity=0.7,
    showlegend=True,
    name='MACD Signal'
), row=6, col=1)

fig_btc_monthly.add_trace(go.Bar(
    x=btc_macd_values_monthly['histogram'].index,
    y=btc_macd_values_monthly['histogram'],
    marker_color=btc_macd_values_monthly['histogram'],
    showlegend=True,
    name='MACD Histogram'
), row=6, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_macd_values_monthly['macd'][btc_macd_buy_monthly].index,
    y=btc_macd_values_monthly['macd'][btc_macd_buy_monthly],
    mode='markers',
    marker=dict(size=12,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='MACD Buy Signals'
), row=6, col=1)

fig_btc_monthly.add_trace(go.Scatter(
    x=btc_macd_values_monthly['macd'][btc_macd_sell_monthly].index,
    y=btc_macd_values_monthly['macd'][btc_macd_sell_monthly],
    mode='markers',
    marker=dict(size=12,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='MACD Sell Signals'
), row=6, col=1)


fig_btc_monthly.update_layout(
    title='BTC-USD Monthly',
    height=1200,
    width=1600,
    showlegend=True,
    xaxis_rangeslider_visible=False,
    template='plotly_dark'
)

fig_btc_monthly.show()


Figure Spy Weekly

In [ ]:
fig_spy_weekly = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.0008,
    subplot_titles=('SPY Weekly'),
    row_width=[1.0, 1.0, 1.0]
)

'''fig_spy_weekly.add_trace(go.Candlestick(
    x=spy_weekly_df.index,
    open=spy_weekly_df['Open']['SPY'],
    high=spy_weekly_df['High']['SPY'],
    low=spy_weekly_df['Low']['SPY'],
    close=spy_weekly_df['Close']['SPY'],
    name='SPY',

), row=1, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_weekly_df.index[spy_buy_confirmed_weekly],
    y=spy_weekly_df['Close']['SPY'][spy_buy_confirmed_weekly],
    mode='markers',
    marker=dict(size=15,
                color='#00BFFF',
                symbol='star'),
    showlegend=True,
    name='SPY Buy Confirmed Weekly'
), row=1, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_weekly_df.index[spy_sell_confirmed_weekly],
    y=spy_weekly_df['Close']['SPY'][spy_sell_confirmed_weekly],
    mode='markers',
    marker=dict(size=15,
                color='#FFFF00',
                colorscale='PuRd',
                symbol='star'),
    showlegend=True,
    name='SPY Sell Confirmed Weekly'
), row=1, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_weekly_df.index[spy_buy_combined_weekly],
    y=spy_weekly_df['Close']['SPY'][spy_buy_combined_weekly],
    mode='markers',
    marker=dict(size=15,
                color='blue',
                colorscale='PuRd',
                symbol='arrow-up'),
    opacity=0.7,
    showlegend=True,
    name='Spy Buy Combined Signals'
), row=1, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_weekly_df.index[spy_sell_combined_weekly],
    y=spy_weekly_df['Close']['SPY'][spy_sell_combined_weekly],
    mode='markers',
    marker=dict(size=15,
                color='red',
                colorscale='PuRd',
                symbol='arrow-down'),
    opacity=0.7,
    showlegend=True,
    name='Spy Sell Combined Signals'
), row=1, col=1)'''

#---------------------------RSI------------------------#

'''fig_spy_weekly.add_trace(go.Scatter(
    x=spy_weekly_rsi_values.index,
    y=spy_weekly_rsi_values,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='RSI'
), row=2, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_weekly_rsi_values[spy_weekly_rsi_buy].index,
    y=spy_weekly_rsi_values[spy_weekly_rsi_buy],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='Spy RSI Buy Signals'
), row=2, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_weekly_rsi_values[spy_weekly_rsi_sell].index,
    y=spy_weekly_rsi_values[spy_weekly_rsi_sell],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='Spy RSI Sell Signals'
), row=2, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_weekly_rsi_sma_20.index,
    y=spy_weekly_rsi_sma_20,
    line=dict(color='orange', width=2),
    opacity=0.7,
    showlegend=True,
    name='SMA 20'
), row=2, col=1)

#--------------------------STOCH RSI---------------------#

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_weekly_rsi_values.index,
    y=spy_weekly_rsi_values,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='RSI'
), row=3, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=rsi_stochrsi_values_weekly['d'].index,
    y=rsi_stochrsi_values_weekly['d'],
    line=dict(color='lightcoral', width=2),
    showlegend=True,
    name='Spy StochRSI d'
), row=3, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=rsi_stochrsi_values_weekly['d'][spy_rsi_stochrsi_buy].index,
    y=rsi_stochrsi_values_weekly['d'][spy_rsi_stochrsi_buy],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='Spy StochRSI Buy Signals'
), row=3, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=rsi_stochrsi_values_weekly['d'][spy_rsi_stochrsi_sell].index,
    y=rsi_stochrsi_values_weekly['d'][spy_rsi_stochrsi_sell],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='Spy StochRSI Sell Signals'
), row=3, col=1)'''

#--------------------BB%B-----------------------#

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_bb_values_weekly.index,
    y=spy_bb_values_weekly,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='BB'
), row=1, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_bb_values_weekly[spy_bb_buy_weekly].index,
    y=spy_bb_values_weekly[spy_bb_buy_weekly],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='Spy BB Buy Signals'
), row=1, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_bb_values_weekly.index[spy_bb_sell_weekly],
    y=spy_bb_values_weekly[spy_bb_sell_weekly],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='Spy BB Sell Signals'
), row=1, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=spy_bb_sma_20_weekly.index,
    y=spy_bb_sma_20_weekly,
    line=dict(color='orange', width=2),
    opacity=0.7,
    showlegend=True,
    name='BB SMA 20'
), row=1, col=1)

fig_spy_weekly.add_hline(y=0, line_dash="dot", line_color="lightgreen", row=1, col=1)
fig_spy_weekly.add_hline(y=0.50, line_dash="dot", line_color="gray", row=1, col=1)
fig_spy_weekly.add_hline(y=1, line_dash="dot", line_color="lightcoral", row=1, col=1)

#------------------------MACD------------------------#

'''fig_spy_weekly.add_trace(go.Scatter(
    x=spy_macd_values_weekly['macd'].index,
    y=spy_macd_values_weekly['macd'],
    line=dict(color='lightblue', width=1),
    showlegend=True,
    name='Spy MACD'
), row=5, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_macd_values_weekly['signal'].index,
    y=spy_macd_values_weekly['signal'],
    line=dict(color='red', width=1),
    opacity=0.7,
    showlegend=True,
    name='Spy MACD Signal'
), row=5, col=1)

fig_spy_weekly.add_trace(go.Bar(
    x=spy_macd_values_weekly['histogram'].index,
    y=spy_macd_values_weekly['histogram'],
    marker_color=spy_macd_values_weekly['histogram'],
    showlegend=True,
    name='Spy MACD Histogram'
), row=5, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=spy_macd_values_weekly['macd'][spy_macd_buy_weekly].index,
    y=spy_macd_values_weekly['macd'][spy_macd_buy_weekly],
    mode='markers',
    marker=dict(size=12,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='MACD Buy Signals'
), row=5, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=spy_macd_values_weekly['macd'][spy_macd_sell_weekly].index,
    y=spy_macd_values_weekly['macd'][spy_macd_sell_weekly],
    mode='markers',
    marker=dict(size=12,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='Spy MACD Sell Signals'
), row=5, col=1)'''

#------------------------SHARPE-----------------------#


fig_spy_weekly.add_trace(go.Scatter(
    x=spy_sharpe_values.index,
    y=spy_sharpe_values,
    line=dict(color='lightblue', width=1),
    showlegend=True,
    name='Sharpe'
), row=2, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_sharpe_values[spy_sharpe_buy_weekly].index,
    y=spy_sharpe_values[spy_sharpe_buy_weekly],
    mode='markers',
    marker=dict(size=spy_sharpe_values[spy_sharpe_buy_weekly].abs()*15,
                color='green',
                symbol='circle'),

    showlegend=True,
    opacity=0.5,
    name='Spy Sharpe Buy Signals'
), row=2, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_sharpe_values[spy_sharpe_sell_weekly].index,
    y=spy_sharpe_values[spy_sharpe_sell_weekly],
    mode='markers',
    marker=dict(size=spy_sharpe_values[spy_sharpe_sell_weekly].abs()*4,
                color='red',
                symbol='circle'),

    showlegend=True,
    opacity=0.7,
    name='Spy Sharpe Sell Signals'
), row=2, col=1)

fig_btc_weekly.add_trace(go.Scatter(
    x=spy_sharpe_sma_20_weekly.index,
    y=spy_sharpe_sma_20_weekly,
    line=dict(color='purple', width=2),
    showlegend=True,
    opacity=0.7,
    name='Spy Sharpe SMA 20'
), row=2, col=1)

#------------------------SORTINO-----------------------#

fig_spy_weekly.add_trace(go.Bar(
    x=spy_sortino_values.index,
    y=spy_sortino_values,
    marker_color=spy_sortino_values,
    showlegend=True,
    opacity=0.7,
    name='Sortino'
), row=2, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_sortino_values[spy_sortino_buy_weekly].index,
    y=spy_sortino_values[spy_sortino_buy_weekly],
    mode='markers',
    marker=dict(size=spy_sortino_values[spy_sortino_buy_weekly].abs()*5,
                color='green',
                symbol='triangle-up'),

    showlegend=True,
    opacity=0.7,
    name='Spy Sortino Buy Signals'
), row=2, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_sortino_values[spy_sortino_sell_weekly].index,
    y=spy_sortino_values[spy_sortino_sell_weekly],
    mode='markers',
    marker=dict(size=spy_sortino_values[spy_sortino_sell_weekly].abs()*3,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='Spy Sortino Sell Signals'
), row=2, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_sortino_sma_14_weekly.index,
    y=spy_sortino_sma_14_weekly,
    line=dict(color='#FFBF00', width=2),
    showlegend=True,
    opacity=0.5,
    name='Spy Sortino SMA 14'
), row=2, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=spy_sortino_sma_7_weekly.index,
    y=spy_sortino_sma_7_weekly,
    line=dict(color='#FF4500', width=2),
    showlegend=True,
    opacity=0.5,
    name='Spy Sortino SMA 7'
), row=2, col=1)

#------------------------VIX----------------------#

fig_spy_weekly.add_trace(go.Bar(
    x=vix.index,
    y=vix,
    marker_color=vix,
    showlegend=True,
    opacity=0.7,
    name='VIX'
), row=3, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=vix_sma_9.index,
    y=vix_sma_9,
    marker_color='orange',
    showlegend=True,
    opacity=0.7,
    name='VIX SMA 9'
), row=3, col=1)

fig_spy_weekly.add_trace(go.Scatter(
    x=vix_sma_20.index,
    y=vix_sma_20,
    marker_color='lightgreen',
    showlegend=True,
    name='VIX SMA 50'
), row=3, col=1)

fig_spy_weekly.update_layout(xaxis_rangeslider_visible=False, template='plotly_dark', height=1000, width=1400)

Figure Spy Monthly

In [ ]:
fig_spy_monthly = make_subplots(
    rows=5, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.008,
    subplot_titles=('Spy Monthly'),
    row_width=[1.0, 1.0, 1.0, 1.0, 1.0],
    specs=[[{"secondary_y": True}], [{"secondary_y": True}], [{"secondary_y": True}], [{"secondary_y": True}], [{"secondary_y": True}]]
)

'''fig_spy_monthly.add_trace(go.Candlestick(
    x=spy_df_monthly.index,
    open=spy_df_monthly['Open']['SPY'],
    high=spy_df_monthly['High']['SPY'],
    low=spy_df_monthly['Low']['SPY'],
    close=spy_df_monthly['Close']['SPY'],
    name='Spy Monthly',

), row=1, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_df_monthly.index[spy_buy_sharpe_sortino_confirmed_monthly],
    y=spy_df_monthly['Close']['SPY'][spy_buy_sharpe_sortino_confirmed_monthly],
    mode='markers',
    marker=dict(size=15,
                color='#00BFFF',
                symbol='star'),
    showlegend=True,
    name='Buy Very Strong Signals'
), row=1, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_df_monthly.index[spy_sell_sharpe_sortino_confirmed_monthly],
    y=spy_df_monthly['Close']['SPY'][spy_sell_sharpe_sortino_confirmed_monthly],
    mode='markers',
    marker=dict(size=15,
                color='#FFFF00',
                symbol='star'),
    showlegend=True,
    name='Sell Very Strong Signals'
), row=1, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_df_monthly.index[spy_buy_combined_monthly],
    y=spy_df_monthly['Close']['SPY'][spy_buy_combined_monthly],
    mode='markers',
    marker=dict(size=10,
                color='blue',
                colorscale='PuRd',
                symbol='arrow-up'),
    opacity=0.7,
    showlegend=True,
    name='Buy Strong Signals'
), row=1, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_df_monthly.index[spy_sell_combined_monthly],
    y=spy_df_monthly['Close']['SPY'][spy_sell_combined_monthly],
    mode='markers',
    marker=dict(size=10,
                color='red',
                colorscale='PuRd',
                symbol='arrow-down'),
    opacity=0.7,
    showlegend=True,
    name='Sell Strong Signals'
), row=1, col=1)


#------------------------MACD------------------------#


fig_spy_monthly.add_trace(go.Scatter(
    x=spy_df_monthly.index[spy_macd_buy_monthly],
    y=spy_df_monthly['Close']['SPY'][spy_macd_buy_monthly],
    mode='markers',
    marker=dict(size=15,
                color='#4CAF50',
                symbol='arrow-up'),
    showlegend=True,
    name='MACD Buy Signals Monthly'
), row=1, col=1)


fig_spy_monthly.add_trace(go.Scatter(
    x=spy_df_monthly.index[spy_macd_sell_monthly],
    y=spy_df_monthly['Close']['SPY'][spy_macd_sell_monthly],
    mode='markers',
    marker=dict(size=15,
                color='#FF0000',
                symbol='arrow-down'),
    showlegend=True,
    name='MACD Sell Signals Monthly'
), row=1, col=1)'''


#---------------------------------Sharpe Sortino -------------------------------------#


fig_spy_monthly.add_trace(go.Scatter(
    x=spy_sharpe_values_monthly.index,
    y=spy_sharpe_values_monthly,
    line=dict(color='lightblue', width=1),
    showlegend=True,
    name='Sharpe'
), row=1, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_sharpe_values_monthly[spy_sharpe_buy_monthly].index,
    y=spy_sharpe_values_monthly[spy_sharpe_buy_monthly],
    mode='markers',
    marker=dict(size=spy_sharpe_values_monthly[spy_sharpe_buy_monthly].abs()*7,
                color='green',
                symbol='circle'),

    showlegend=True,
    opacity=0.5,
    name='Sharpe Buy Signals'
), row=1, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_sharpe_values_monthly[spy_sharpe_sell_monthly].index,
    y=spy_sharpe_values_monthly[spy_sharpe_sell_monthly],
    mode='markers',
    marker=dict(size=spy_sharpe_values_monthly[spy_sharpe_sell_monthly].abs()*4,
                color='red',
                symbol='circle'),

    showlegend=True,
    opacity=0.7,
    name='Sharpe Sell Signals'
), row=1, col=1)

'''fig_spy_monthly.add_trace(go.Scatter(
    x=spy_sharpe_sma_14_monthly.index,
    y=spy_sharpe_sma_14_monthly,
    line=dict(color='purple', width=2),
    showlegend=True,
    opacity=0.7,
    name='Sharpe SMA 14'
), row=1, col=1)'''

fig_spy_monthly.add_trace(go.Bar(
    x=spy_sortino_values_monthly.index,
    y=spy_sortino_values_monthly,
    marker_color=spy_sortino_values_monthly,
    showlegend=True,
    opacity=0.7,
    name='Sortino'
), row=1, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_sortino_values_monthly[spy_sortino_buy_monthly].index,
    y=spy_sortino_values_monthly[spy_sortino_buy_monthly],
    mode='markers',
    marker=dict(size=spy_sortino_values_monthly[spy_sortino_buy_monthly].abs()*5,
                color='green',
                symbol='triangle-up'),

    showlegend=True,
    opacity=0.7,
    name='Sortino Buy Signals'
), row=1, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_sortino_values_monthly[spy_sortino_sell_monthly].index,
    y=spy_sortino_values_monthly[spy_sortino_sell_monthly],
    mode='markers',
    marker=dict(size=spy_sortino_values_monthly[spy_sortino_sell_monthly].abs()*3,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='Sortino Sell Signals'
), row=1, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_sortino_sma_14_monthly.index,
    y=spy_sortino_sma_14_monthly,
    line=dict(color='#FFBF00', width=2),
    showlegend=True,
    opacity=0.5,
    name='Sortino SMA 14'
), row=1, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_sortino_sma_7_monthly.index,
    y=spy_sortino_sma_7_monthly,
    line=dict(color='#FF4500', width=2),
    showlegend=True,
    opacity=0.5,
    name='Sortino SMA 7'
), row=1, col=1)

#----------------------------RSI-----------------------------#


fig_spy_monthly.add_trace(go.Scatter(
    x=spy_rsi_values_monthly.index,
    y=spy_rsi_values_monthly,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='RSI'
), row=2, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_rsi_values_monthly[spy_rsi_buy_monthly].index,
    y=spy_rsi_values_monthly[spy_rsi_buy_monthly],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='RSI Buy Signals'
), row=2, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_rsi_values_monthly[spy_rsi_sell_monthly].index,
    y=spy_rsi_values_monthly[spy_rsi_sell_monthly],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='RSI Sell Signals'
), row=2, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_rsi_sma_20_monthly.index,
    y=spy_rsi_sma_20_monthly,
    line=dict(color='orange', width=2),
    opacity=0.7,
    showlegend=True,
    name='RSI SMA 20'
), row=2, col=1)

fig_spy_monthly.add_hline(y=20, line_dash="dot", line_color="lightgreen", row=2, col=1)
fig_spy_monthly.add_hline(y=50, line_dash="dot", line_color="gray", row=2, col=1)
fig_spy_monthly.add_hline(y=80, line_dash="dot", line_color="lightcoral", row=2, col=1)


#--------------------------------Stoch Rsi------------------------------------#

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_stochrsi_values_monthly['d'].index,
    y=spy_stochrsi_values_monthly['d'],
    line=dict(color='lightcoral', width=2),
    showlegend=True,
    name='StochRSI d'
), row=3, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_rsi_values_monthly.index,
    y=spy_rsi_values_monthly,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='StochRSI'
), row=3, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_stochrsi_values_monthly['d'][spy_stochrsi_buy_monthly].index,
    y=spy_stochrsi_values_monthly['d'][spy_stochrsi_buy_monthly],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='StochRSI Buy Signals'
), row=3, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_stochrsi_values_monthly['d'][spy_stochrsi_sell_monthly].index,
    y=spy_stochrsi_values_monthly['d'][spy_stochrsi_sell_monthly],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='StochRSI Sell Signals'
), row=3, col=1)

fig_spy_monthly.add_hline(y=10, line_dash="dot", line_color="lightgreen", row=3, col=1)
fig_spy_monthly.add_hline(y=50, line_dash="dot", line_color="gray", row=3, col=1)
fig_spy_monthly.add_hline(y=90, line_dash="dot", line_color="lightcoral", row=3, col=1)

#---------------------------------BB%B----------------------------#

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_bb_values_monthly.index,
    y=spy_bb_values_monthly,
    line=dict(color='lightblue', width=2),
    showlegend=True,
    name='BB%B'
), row=4, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_bb_values_monthly[spy_bb_buy_monthly].index,
    y=spy_bb_values_monthly[spy_bb_buy_monthly],
    mode='markers',
    marker=dict(size=10,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='BB Buy Signals'
), row=4, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_bb_values_monthly[spy_bb_sell_monthly].index,
    y=spy_bb_values_monthly[spy_bb_sell_monthly],
    mode='markers',
    marker=dict(size=10,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='BB Sell Signals'
), row=4, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_bb_sma_20_monthly.index,
    y=spy_bb_sma_20_monthly,
    line=dict(color='orange', width=2),
    opacity=0.7,
    showlegend=True,
    name='BB SMA 20'
), row=4, col=1)

fig_spy_monthly.add_hline(y=0, line_dash="dot", line_color="lightgreen", row=4, col=1)
fig_spy_monthly.add_hline(y=0.50, line_dash="dot", line_color="gray", row=4, col=1)
fig_spy_monthly.add_hline(y=1, line_dash="dot", line_color="lightcoral", row=4, col=1)

#------------------------------MACD--------------------------#

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_macd_values_monthly['macd'].index,
    y=spy_macd_values_monthly['macd'],
    line=dict(color='lightblue', width=1),
    showlegend=True,
    name='MACD'
), row=5, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_macd_values_monthly['signal'].index,
    y=spy_macd_values_monthly['signal'],
    line=dict(color='red', width=1),
    opacity=0.7,
    showlegend=True,
    name='MACD Signal'
), row=5, col=1)

fig_spy_monthly.add_trace(go.Bar(
    x=spy_macd_values_monthly['histogram'].index,
    y=spy_macd_values_monthly['histogram'],
    marker_color=spy_macd_values_monthly['histogram'],
    showlegend=True,
    name='MACD Histogram'
), row=5, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_macd_values_monthly['macd'][spy_macd_buy_monthly].index,
    y=spy_macd_values_monthly['macd'][spy_macd_buy_monthly],
    mode='markers',
    marker=dict(size=12,
                color='green',
                symbol='triangle-up'),
    showlegend=True,
    opacity=0.7,
    name='MACD Buy Signals'
), row=5, col=1)

fig_spy_monthly.add_trace(go.Scatter(
    x=spy_macd_values_monthly['macd'][spy_macd_sell_monthly].index,
    y=spy_macd_values_monthly['macd'][spy_macd_sell_monthly],
    mode='markers',
    marker=dict(size=12,
                color='red',
                symbol='triangle-down'),
    showlegend=True,
    opacity=0.7,
    name='MACD Sell Signals'
), row=5, col=1)


fig_spy_monthly.update_layout(
    title='SPY Monthly',
    height=1200,
    width=1600,
    showlegend=True,
    xaxis_rangeslider_visible=False,
    template='plotly_dark'
)

fig_spy_monthly.show()